# ro-id — generare dataset + antrenare PaddleOCR (Colab)

1. **Generează** imagini sintetice CI (`generate.py`)
2. **Antrenează** modelul (`train_and_deploy.py`)
3. Descarcă **`model_export.zip`**

**Runtime → GPU** (pentru antrenare; generarea merge și pe CPU, dar GPU e recomandat).

Salvează datasetul pe **Google Drive** — spațiul din `/content` se șterge la deconectare.

In [ ]:
# === Configurare (EDITEAZĂ AICI) ===
REPO_URL = "https://github.com/catalingy-cpu/ro-id-syntetic-database.git"
REPO_BRANCH = "main"   # sau "master" dacă pe GitHub branch-ul e master
WORKDIR = "/content/ro-id"

# Repo PRIVAT: token pe https://github.com/settings/tokens (bifează "repo")
GITHUB_TOKEN = ""

# Re-descarcă repo la fiecare sesiune (recomandat după push pe GitHub)
FORCE_REFRESH_REPO = True

# --- Dataset ---
GENERATE_DATASET = True           # True = rulează generate.py în Colab
SKIP_GENERATION_IF_EXISTS = True  # sare generarea dacă există labels/train.txt
GENERATE_COUNT = 5000             # 5k–20k rezonabil pe Colab; 50k+ pe Drive cu răbdare
GENERATE_BATCH_SIZE = 200
GENERATE_WORKERS = 2              # ~2 CPU în Colab

# Multi-template: clasic (ROI calibrat) + electronic + telefon
GENERATE_MULTI = True
GENERATE_MIX = "config/generation_mix.json"   # classic_reference + eID + phone
GENERATE_FIELDS = "config/template_fields.json"  # doar dacă GENERATE_MULTI = False

DATASET_DIR = "/content/drive/MyDrive/ro-id/dataset_colab"  # recomandat: Drive
MOUNT_GOOGLE_DRIVE = True
UPLOAD_DATASET_ZIP = False        # alternativ: încarcă ZIP cu dataset deja generat

# --- Antrenare ---
EPOCHS = 15
BATCH_SIZE = 128
USE_FAST = False
SKIP_CHECK_DATASET = False
INSTALL_PADDLEOCR_REPO = True
MODEL_NAME = "frc_ci_rec"

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

def run(cmd, *, cwd=None, env=None, check=True):
    print(">", " ".join(str(c) for c in cmd))
    return subprocess.run(cmd, cwd=cwd, env=env, check=check)

try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception:
    gpu = None
if gpu:
    print(f"GPU: {gpu}")
else:
    r = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
    print(r.stdout.strip() or "GPU: indisponibil (generarea merge pe CPU)")

In [ ]:
import re
import zipfile
import urllib.request

def _parse_github_repo(url: str) -> tuple[str, str]:
    url = url.strip().rstrip("/")
    if url.endswith(".git"):
        url = url[:-4]
    m = re.match(r"https?://github\.com/([^/]+)/([^/]+)$", url)
    if not m:
        raise SystemExit(f"REPO_URL invalid: {url}")
    return m.group(1), m.group(2)

def _clone_url(url: str) -> str:
    url = url.strip()
    if not GITHUB_TOKEN:
        return url
    if url.startswith("https://") and "@" not in url.split("//", 1)[1]:
        return url.replace("https://", f"https://{GITHUB_TOKEN}@", 1)
    return url

def _download_zip(owner: str, repo: str, branch: str, dest: Path) -> None:
    zip_url = f"https://github.com/{owner}/{repo}/archive/refs/heads/{branch}.zip"
    zip_path = Path("/tmp") / f"{repo}-{branch}.zip"
    print(f"Descărcare ZIP: {zip_url}")
    req = urllib.request.Request(zip_url, headers={"User-Agent": "colab-ro-id"})
    with urllib.request.urlopen(req) as resp, zip_path.open("wb") as out:
        shutil.copyfileobj(resp, out)
    extract_tmp = Path("/tmp") / f"{repo}-extract"
    if extract_tmp.exists():
        shutil.rmtree(extract_tmp)
    extract_tmp.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_tmp)
    extracted = next(extract_tmp.iterdir())
    if dest.exists():
        shutil.rmtree(dest)
    shutil.move(str(extracted), str(dest))
    zip_path.unlink(missing_ok=True)
    shutil.rmtree(extract_tmp, ignore_errors=True)
    print(f"Repo extras în {dest}")

workdir = Path(WORKDIR)
if FORCE_REFRESH_REPO and workdir.is_dir():
    shutil.rmtree(workdir)
    print(f"Șters repo vechi: {WORKDIR}")

if workdir.is_dir() and (workdir / "generate.py").is_file():
    print(f"Repo există deja: {WORKDIR}")
else:
    if workdir.is_dir():
        shutil.rmtree(workdir)
    owner, repo = _parse_github_repo(REPO_URL)
    clone_url = _clone_url(REPO_URL.strip())
    print(f"Clone git → {WORKDIR}")
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, clone_url, WORKDIR],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print("git clone eșuat:", (r.stderr or r.stdout).strip())
        print("Încerc descărcare ZIP (merge adesea când git eșuează pe Colab)...")
        _download_zip(owner, repo, REPO_BRANCH, workdir)

REPO_ROOT = Path(WORKDIR).resolve()
COLAB_ROOT = REPO_ROOT / "colab"
assert (REPO_ROOT / "generate.py").is_file(), "Lipsește generate.py"
assert (REPO_ROOT / "scripts" / "train_and_deploy.py").is_file()
need = REPO_ROOT / "ro_id_synth" / "template_registry.py"
if not need.is_file():
    raise SystemExit(
        "Repo incomplet (lipsește template_registry.py).\n"
        "Push codul nou pe GitHub, apoi FORCE_REFRESH_REPO=True și re-rulează această celulă."
    )
print("Repo:", REPO_ROOT)

In [ ]:
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Fonturi Linux pentru randare text CI (fără Windows Fonts)
subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "fonts-liberation", "fontconfig"],
    check=False,
)

# Colab: folosim Python-ul din sesiune (venv în /content dă erori pe Colab)
VENV_PY = Path(sys.executable)
print("Python:", VENV_PY)

run([str(VENV_PY), "-m", "pip", "install", "-U", "pip", "wheel", "setuptools"])
run([str(VENV_PY), "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements.txt")])
print("Dependențe generator instalate.")

In [ ]:
dataset_path = Path(DATASET_DIR)
train_labels = dataset_path / "labels" / "train.txt"

if UPLOAD_DATASET_ZIP:
    from google.colab import files
    zip_name = next(iter(files.upload().keys()))
    extract_to = REPO_ROOT / "dataset_upload"
    if extract_to.exists():
        shutil.rmtree(extract_to)
    extract_to.mkdir(parents=True)
    shutil.unpack_archive(zip_name, extract_to)
    if (extract_to / "dataset" / "labels" / "train.txt").is_file():
        dataset_path = extract_to / "dataset"
    elif (extract_to / "labels" / "train.txt").is_file():
        dataset_path = extract_to
    else:
        raise SystemExit("ZIP fără labels/train.txt")
    train_labels = dataset_path / "labels" / "train.txt"

elif GENERATE_DATASET:
    if SKIP_GENERATION_IF_EXISTS and train_labels.is_file():
        n = sum(1 for ln in train_labels.read_text(encoding="utf-8").splitlines() if ln.strip())
        print(f"Dataset existent pe Drive ({n} etichete) — sar generarea.")
    else:
        import json
        if GENERATE_MULTI:
            mix_cfg = (REPO_ROOT / GENERATE_MIX).resolve()
            if not mix_cfg.is_file():
                raise SystemExit(f"Lipsește {mix_cfg} — repo vechi, fă push + FORCE_REFRESH_REPO=True")
            print("Multi-template:", mix_cfg.name)
        else:
            fields_cfg = (REPO_ROOT / GENERATE_FIELDS).resolve()
            if not fields_cfg.is_file():
                raise SystemExit(f"Lipsește {fields_cfg}")
            tpl = REPO_ROOT / json.loads(fields_cfg.read_text(encoding="utf-8"))["template_image"]
            if not tpl.is_file():
                specimens = sorted((REPO_ROOT / "templates").glob("*.png"))
                hint = ", ".join(p.name for p in specimens[:5]) or "(niciun PNG în templates/)"
                raise SystemExit(
                    f"Template lipsă: {tpl}\n"
                    f"Specimen găsite în repo: {hint}\n"
                    "Comite templates/*.png specimen în repo GitHub înainte de Colab."
                )
        dataset_path.mkdir(parents=True, exist_ok=True)
        est_min = max(10, GENERATE_COUNT // 80)
        gen_cmd = [
            str(VENV_PY), str(REPO_ROOT / "generate.py"),
            "--count", str(GENERATE_COUNT),
            "--output", str(dataset_path),
            "--batch-size", str(GENERATE_BATCH_SIZE),
            "--workers", str(GENERATE_WORKERS),
        ]
        if GENERATE_MULTI:
            mix_cfg = (REPO_ROOT / GENERATE_MIX).resolve()
            if not mix_cfg.is_file():
                raise SystemExit(f"Lipsește {mix_cfg}")
            gen_cmd += ["--multi", "--mix", str(mix_cfg)]
            print(
                f"Generare multi-template {GENERATE_COUNT} imagini → {dataset_path}\n"
                f"  mix: {mix_cfg.name} | workers: {GENERATE_WORKERS} | ~{est_min}–{est_min * 2} min"
            )
        else:
            gen_cmd += ["--fields", str(fields_cfg)]
            print(
                f"Generare {GENERATE_COUNT} imagini → {dataset_path}\n"
                f"  template: {tpl.name} | workers: {GENERATE_WORKERS} | ~{est_min}–{est_min * 2} min"
            )
        run(gen_cmd, cwd=REPO_ROOT)
        train_labels = dataset_path / "labels" / "train.txt"

if not train_labels.is_file():
    raise SystemExit(
        f"Lipsește {train_labels}. Setează GENERATE_DATASET=True sau UPLOAD_DATASET_ZIP=True."
    )
if not (dataset_path / "images").is_dir():
    raise SystemExit(f"Lipsește {dataset_path / 'images'}")

n_labels = sum(1 for ln in train_labels.read_text(encoding="utf-8").splitlines() if ln.strip())
print(f"Dataset gata: {dataset_path} ({n_labels} sample-uri)")

In [ ]:
# Paddle + dependențe antrenare
# NU folosi cu123 — Colab 2025+ necesită cu126/cu129
import re

def install_paddle_colab(py: str) -> None:
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    m = re.search(r"CUDA Version:\s*(\d+\.\d+)", r.stdout or "")
    cuda = m.group(1) if m else None
    print(f"CUDA (driver): {cuda or 'necunoscut'} — Runtime trebuie să fie GPU")
    urls = [
        "https://www.paddlepaddle.org.cn/packages/stable/cu126/",
        "https://www.paddlepaddle.org.cn/packages/stable/cu129/",
        "https://www.paddlepaddle.org.cn/packages/stable/cu118/",
    ]
    last = ""
    for url in urls:
        for ver in ("3.3.0", "3.2.2", "3.2.0"):
            cmd = [py, "-m", "pip", "install", "-U", f"paddlepaddle-gpu=={ver}", "-i", url]
            print(">", " ".join(cmd))
            proc = subprocess.run(cmd, capture_output=True, text=True)
            if proc.returncode == 0:
                print(f"OK paddlepaddle-gpu=={ver}")
                return
            last = (proc.stderr or proc.stdout)[-600:]
    raise SystemExit(f"paddlepaddle-gpu eșuat.\n{last}")

install_paddle_colab(str(VENV_PY))
run([str(VENV_PY), "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements-train.txt")])
run([str(VENV_PY), "-m", "pip", "install", "paddleocr>=3.0.0,<4.0.0"])
run([str(VENV_PY), "-c", "import paddle; print('Paddle', paddle.__version__, 'CUDA', paddle.is_compiled_with_cuda())"])

In [ ]:
dataset_rel = os.path.relpath(dataset_path.resolve(), REPO_ROOT)
cmd = [str(VENV_PY), str(REPO_ROOT / "scripts" / "train_and_deploy.py"),
       "--dataset", dataset_rel, "--device", "gpu:0",
       "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
       "--paddle-python", str(VENV_PY), "--model-name", MODEL_NAME,
       "--skip-deploy"]
if USE_FAST: cmd.append("--fast")
if SKIP_CHECK_DATASET: cmd.append("--skip-check-dataset")
if INSTALL_PADDLEOCR_REPO: cmd.append("--install-paddleocr-repo")

env = os.environ.copy()
env.update({"PADDLE_DEVICE": "gpu:0", "FLAGS_use_mkldnn": "0", "MPLBACKEND": "Agg",
            "PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK": "True"})
run(cmd, cwd=REPO_ROOT, env=env)

In [ ]:
sys.path.insert(0, str(COLAB_ROOT))
from package_model_export import package_model_export

MODEL_DIR = REPO_ROOT / "exports" / MODEL_NAME
ZIP_PATH = COLAB_ROOT / "model_export.zip"
package_model_export(model_dir=MODEL_DIR, zip_path=ZIP_PATH)
print(f"{ZIP_PATH} ({ZIP_PATH.stat().st_size / 1_048_576:.1f} MB)")

In [ ]:
from google.colab import files
files.download(str(ZIP_PATH))
print("Copiază model_export.zip în FRCHub/services/paddle-ocr/models/")

## Note

- **Prima rulare:** generează + antrenează (ore, în funcție de `GENERATE_COUNT` și `EPOCHS`).
- **Rulări următoare:** cu `SKIP_GENERATION_IF_EXISTS=True`, refolosește datasetul de pe Drive.
- Template-urile specimen sunt în repo (`templates/*.png`) — nu ai nevoie de PC pentru generare.
- Pentru 50k imagini: crește `GENERATE_COUNT`, asigură spațiu pe Drive (~10–30 GB).